# Coordinate-based Tile Extraction from WSI

This notebook demonstrates how to extract tiles from whole slide images at specific coordinates using the MINOTAUR tiling system.

## Features:
- Extract tiles at specific coordinates
- Support for multiple coordinates
- Optional stain normalization
- Border checking to ensure tiles fit within slide boundaries
- Easy integration with existing MINOTAUR infrastructure


## Setup and Imports


In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Add the tiler directory to the path
sys.path.append('/Users/w2030634/Projects/MINOTAUR/MINOTAUR/cem_mil/tiler')

from coordinate_tile_extractor import CoordinateTileExtractor, create_stain_normalizer


## Basic Usage - Extract Single Tile


In [ ]:
# Define your WSI path
wsi_path = "/path/to/your/slide.svs"  # Replace with your actual WSI path

# Define coordinates (x, y) at level 0 (highest resolution)
x_coord = 10000  # Example x coordinate
y_coord = 15000  # Example y coordinate

# Initialize the extractor
extractor = CoordinateTileExtractor(
    wsi_path=wsi_path,
    tile_size=512,  # Size of tiles to extract
    mag_level=0     # Magnification level (0 = highest resolution)
)

# Extract a single tile
tile = extractor.extract_tile_at_coordinates(x_coord, y_coord)

# Display the tile
plt.figure(figsize=(8, 8))
plt.imshow(tile)
plt.title(f'Tile at coordinates ({x_coord}, {y_coord})')
plt.axis('off')
plt.show()

print(f"Tile shape: {tile.shape}")
print(f"Tile dtype: {tile.dtype}")


## Extract Multiple Tiles


In [ ]:
# Define multiple coordinates
coordinates = [
    (10000, 15000),  # First tile
    (20000, 15000),  # Second tile
    (10000, 25000),  # Third tile
    (20000, 25000),  # Fourth tile
]

# Extract multiple tiles
tiles = extractor.extract_multiple_tiles_at_coordinates(coordinates)

# Display all tiles
fig, axes = plt.subplots(2, 2, figsize=(12, 12))
axes = axes.flatten()

for i, ((x, y), tile) in enumerate(tiles):
    axes[i].imshow(tile)
    axes[i].set_title(f'Tile at ({x}, {y})')
    axes[i].axis('off')

plt.tight_layout()
plt.show()

print(f"Extracted {len(tiles)} tiles")


## Complete Example Function


In [ ]:
def extract_tiles_at_coordinates(
    wsi_path: str,
    coordinates: list,
    output_dir: str = "./extracted_tiles",
    tile_size: int = 512,
    mag_level: int = 0,
    save_tiles: bool = True,
    display_tiles: bool = True
):
    """
    Complete function to extract tiles at specific coordinates.
    
    Args:
        wsi_path: Path to the WSI file
        coordinates: List of (x, y) coordinate tuples
        output_dir: Directory to save tiles (if save_tiles=True)
        tile_size: Size of tiles to extract
        mag_level: Magnification level for extraction
        save_tiles: Whether to save tiles to disk
        display_tiles: Whether to display tiles in notebook
    
    Returns:
        List of (coordinates, tile) tuples
    """
    # Initialize extractor
    extractor = CoordinateTileExtractor(
        wsi_path=wsi_path,
        tile_size=tile_size,
        mag_level=mag_level
    )
    
    # Extract tiles with border checking
    tiles_with_check = extractor.extract_tiles_with_border_check(coordinates)
    
    # Filter valid tiles
    valid_tiles = [(coords, tile) for coords, tile, is_valid in tiles_with_check if is_valid]
    
    print(f"Extracted {len(valid_tiles)} valid tiles out of {len(coordinates)} requested")
    
    # Save tiles if requested
    if save_tiles and valid_tiles:
        saved_paths = extractor.save_tiles(valid_tiles, output_dir)
        print(f"Saved tiles to: {output_dir}")
    
    # Display tiles if requested
    if display_tiles and valid_tiles:
        n_tiles = len(valid_tiles)
        cols = min(4, n_tiles)
        rows = (n_tiles + cols - 1) // cols
        
        fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 4*rows))
        if rows == 1:
            axes = [axes] if cols == 1 else axes
        else:
            axes = axes.flatten()
        
        for i, ((x, y), tile) in enumerate(valid_tiles):
            if i < len(axes):
                axes[i].imshow(tile)
                axes[i].set_title(f'Tile at ({x}, {y})')
                axes[i].axis('off')
        
        # Hide unused subplots
        for i in range(len(valid_tiles), len(axes)):
            axes[i].axis('off')
        
        plt.tight_layout()
        plt.show()
    
    # Close extractor
    extractor.close()
    
    return valid_tiles

# Example usage:
# tiles = extract_tiles_at_coordinates(
#     wsi_path="/path/to/your/slide.svs",
#     coordinates=[(10000, 15000), (20000, 15000), (10000, 25000)],
#     output_dir="./my_tiles",
#     tile_size=512
# )
